
# Avocado Prices adathalmaz elemzése

**A kutatás során ezekre a kérdésekre keressük a válaszokat.**

1. **Mennyire jelezhető előre az avokádó ára trend, szezonalitás és régiós eltérések alapján, és mely időszakokban romlik el látványosan az előrejelzés?** *(Három egyszerű módszerrel becsüljük meg a hagyományos avokádó következő heti árát.)*


2. **Mekkora a bioavokádó felára, és mennyire állandó a régiók és az évek között?** *(Ugyanazon a héten és ugyanabban a régióban hasonlítjuk össze a két árat. A különbséget dollárban és százalékban is megadjuk.)*


3. **Mennyire mozog együtt a hagyományos és a bioavokádó ára?** *(Megnézzük, hasonló időszakokban drága-e a két típus, és azt is, hogy egyik hétről a másikra általában azonos irányba változik-e az áruk.)*


## Adatok értelmezése és előkészítése:

**Forrás:** Justin Kiggins *Avocado Prices* adathalmazát használom, amely a **Hass Avocado Board** amerikai bolti eladási adatain alapul. Egy sorban egy piac, heti avokádi eladási adati vannak (usd/db ár, bio e ?, méret, milyen kiszerelésben) Az adatok 2015 januárjától 2018 márciusái tartanak.

Egy sor egy adott hét, terület és avokádótípus adatait tartalmazza. A `region` oszlopban városi piacok, nagyobb régiók és a `TotalUS` országos összesítés is szerepel.

### Adathalmaz változói:

| Változó | Magyarázat | Példa (első sor értékei) |
|---|---|---:|
| `Date` | A megfigyelési hét, ez mindig vasárnap. | `2015-12-27` |
| `region` | A piac vagy területi neve (állam is lehet, több várost egybefogó piac, vagy régiós összesítés). | `Albany` |
| `type` | Típus. `conventional` = hagyományos, `organic` = bio. | `conventional` |
| `year` | Külön eltárolva év. | `2015` |
| `AveragePrice` | Avokádó átlagos eladási ára, (usd/db). | 1,33 |
| `Total Volume` | Adott piacon, a teljes eladott mennyiség (minden méretből és zacskóból). | 64 236,62 |
| `4046` | Kis méretű, darabonként értékesített mennyiség. | 1 036,74 |
| `4225` | Nagy méretű, darabonként értékesített mennyiség. | 54 454,85 |
| `4770` | Extra nagy méretű, darabonként értékesített mennyiség. | 48,16 |
| `Total Bags` | Zacskós kiszerelésben értékesített összes mennyiség. | 8 696,87 |
| `Small Bags` | Kis zacskós kiszerelésben mennyiség. | 8 603,62 |
| `Large Bags` | Nagy zacskós kiszerelésben mennyiség. | 93,25 |
| `XLarge Bags` | Extra nagy zacskós kiszerelésben mennyiség. | 0 |
| `Unnamed: 0` | A CSV-be mentett régi sorindex, az értékei ismétlődnek, az elemzésből elhagyható. | 0 |

Először ellenőrizzük, hogy minden héthez van-e adat, és előfordul-e, hogy ugyanaz a hét, régió és típus kétszer szerepel. Átnézzük az árak tartományát is. A CSV-ben lévő `Unnamed: 0` csak sorszám, ezért eltávolítjuk.

In [6]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib import dates as mdates
from sklearn.linear_model import LinearRegression

DATA_PATH = Path("data/avocado.csv")
raw = pd.read_csv(DATA_PATH, parse_dates=["Date"])
data = raw.drop(columns=["Unnamed: 0"], errors="ignore")
data = data.sort_values(["region", "type", "Date"]).reset_index(drop=True)

plt.rcParams.update({
    "figure.dpi": 115,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.axisbelow": True,
    "grid.alpha": 0.25,
    "legend.frameon": False,
})
BLUE, ORANGE, TEAL, GRAY = "#22558a", "#b34a21", "#13766d", "#5b6574"
MONTH_NAMES = ["Jan.", "Febr.", "Márc.", "Ápr.", "Máj.", "Jún.",
               "Júl.", "Aug.", "Szept.", "Okt.", "Nov.", "Dec."]

print(f"Adatforrás: {DATA_PATH} | eredeti méret: {raw.shape[0]:,} sor, {raw.shape[1]} oszlop")
print(f"Időszak: {data['Date'].min():%Y-%m-%d} – {data['Date'].max():%Y-%m-%d}")
print(f"Területi címkék: {data['region'].nunique()} | különböző hetek: {data['Date'].nunique()}")
display(data.head(10))
display(data[["AveragePrice"]].describe().round(2))

Adatforrás: data/avocado.csv | eredeti méret: 18,249 sor, 14 oszlop
Időszak: 2015-01-04 – 2018-03-25
Területi címkék: 54 | különböző hetek: 169


,Date,AveragePrice,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,type,year,region
0,2015-01-04,1.22,40873.28,2819.50,28287.42,49.90,9716.46,9186.93,529.53,0.0,conventional,2015,Albany
1,2015-01-11,1.24,41195.08,1002.85,31640.34,127.12,8424.77,8036.04,388.73,0.0,conventional,2015,Albany
2,2015-01-18,1.17,44511.28,914.14,31540.32,135.77,11921.05,11651.09,269.96,0.0,conventional,2015,Albany
3,2015-01-25,1.06,45147.50,941.38,33196.16,164.14,10845.82,10103.35,742.47,0.0,conventional,2015,Albany
4,2015-02-01,0.99,70873.60,1353.90,60017.20,179.32,9323.18,9170.82,152.36,0.0,conventional,2015,Albany
5,2015-02-08,0.99,51253.97,1357.37,39111.81,163.25,10621.54,10113.10,508.44,0.0,conventional,2015,Albany
6,2015-02-15,1.06,41567.62,986.66,30045.51,222.42,10313.03,9979.87,333.16,0.0,conventional,2015,Albany
7,2015-02-22,1.07,45675.05,1088.38,35056.13,151.00,9379.54,9000.16,379.38,0.0,conventional,2015,Albany
8,2015-03-01,0.99,55595.74,629.46,45633.34,181.49,9151.45,8986.06,165.39,0.0,conventional,2015,Albany
9,2015-03-08,1.07,40507.36,795.68,30370.64,159.05,9181.99,8827.55,354.44,0.0,conventional,2015,Albany


,AveragePrice
count,18249.00
mean,1.41
std,0.40
min,0.44
25%,1.10
50%,1.37
75%,1.66
max,3.25


In [7]:
KEY = ["Date", "region", "type"]
calendar_weeks = pd.date_range(data["Date"].min(), data["Date"].max(), freq="W-SUN")
expected = pd.MultiIndex.from_product(
    [calendar_weeks, sorted(data["region"].unique()), sorted(data["type"].unique())],
    names=KEY,
)
observed = pd.MultiIndex.from_frame(data[KEY])
missing_rows = expected.difference(observed)

checks = pd.Series({
    "Hiányzó cellák": data.isna().sum().sum(),
    "Ismétlődő hét–régió–típus kulcsok": data.duplicated(KEY).sum(),
    "Hiányzó hét–régió–típus kombinációk": len(missing_rows),
    "Vasárnaptól eltérő dátumok": data["Date"].dt.dayofweek.ne(6).sum(),
    "Eltérések a dátum és az év között": data["Date"].dt.year.ne(data["year"]).sum(),
    "Nulla vagy negatív árak": data["AveragePrice"].le(0).sum(),
}, name="Darab")
display(checks.to_frame())
display(missing_rows.to_frame(index=False))
display(data.groupby(["year", "type"])["Date"].nunique().unstack().rename_axis(index="Év"))

SELECTED_REGIONS = [
    "California", "GreatLakes", "Midsouth", "Northeast",
    "Plains", "SouthCentral", "Southeast", "West",
]
regional = data.loc[data["region"].isin(SELECTED_REGIONS)].copy()
selected_index = pd.MultiIndex.from_product(
    [calendar_weeks, SELECTED_REGIONS, ["conventional", "organic"]], names=KEY
)
assert data.duplicated(KEY).sum() == 0
assert len(regional) == len(selected_index)
assert selected_index.difference(observed).empty
assert regional["AveragePrice"].gt(0).all()
print(f"A nyolc kiválasztott régió teljes mintája: {len(regional):,} sor.")

,Darab
Hiányzó cellák,0
Ismétlődő hét–régió–típus kulcsok,0
Hiányzó hét–régió–típus kombinációk,3
Vasárnaptól eltérő dátumok,0
Eltérések a dátum és az év között,0
Nulla vagy negatív árak,0


,Date,region,type
0,2015-12-06,WestTexNewMexico,organic
1,2017-06-18,WestTexNewMexico,organic
2,2017-06-25,WestTexNewMexico,organic


type,conventional,organic
Év,,
2015,52,52
2016,52,52
2017,53,53
2018,12,12


A nyolc kiválasztott régió teljes mintája: 2,704 sor.


A fájl meglévő soraiban nincsenek üres cellák, és minden hét/régió/típus kombináció egyszer szerepel. Három héten, a `WestTexNewMexico` bio avokádójáról teljesen hiányzik a sor.

A három vizsgálathoz nyolc nagyobb régiót választunk. Ezekben mindkét típus ára mind a 169 hétre megvan, így azonos időszakokat tudunk vizsgálni. A régiók egyforma súlyt kapnak az átlagokban. Az országos árat közvetlenül a fájl `TotalUS` soraiból olvassuk ki.

A számítások alapja a heti ár. Néhány ábrán hónapok szerint foglaljuk össze a heteket, hogy könnyebb legyen felismerni a mintákat. Egy hét ahhoz a hónaphoz tartozik, amelyikbe a vasárnapja esik. A 2018-as adatok márciusig tartanak. A teljes éveket összehasonlító ábrák ezért 2015-től 2017-ig tartanak.